# Saheli — Unsloth Fine-Tuning on Kaggle (Gemma 4 E4B)

**Run cells in order. After the install cell, restart the kernel, then continue from Step 2.**

### What to upload to your Kaggle Dataset
Upload the entire `saheli/` project folder as a zip (or individual files).
Minimum required files:
- `finetune/train_unsloth.py`
- `data/who_anc_checklist.json`

Optional (for full evaluation):
- `finetune/evaluate_model.py`
- `tests/sample_cases.json`

### Kaggle settings
Settings → Accelerator → **GPU T4 x2**  
Internet → **On** (needed to download model weights)


## Step 1 — Install Unsloth
⚠️ **Restart the kernel after this cell completes**, then run Step 2 onwards.

In [ ]:
# Official Unsloth install for Kaggle T4
# Reference: https://unsloth.ai/docs/models/gemma-4/train
!pip install -q unsloth
!pip install -q --upgrade --no-deps unsloth
!pip install -q trl transformers datasets accelerate peft bitsandbytes huggingface_hub

import torch, transformers, trl, unsloth
print('torch       :', torch.__version__)
print('transformers:', transformers.__version__)
print('trl         :', trl.__version__)
print('unsloth     :', unsloth.__version__)
print('cuda        :', torch.cuda.is_available())
print()
print('*** RESTART KERNEL NOW, then continue from Step 2 ***')

## Step 2 — Set up project directory

In [ ]:
import os, sys, shutil, torch

print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — STOP')
assert torch.cuda.is_available(), 'Enable GPU: Settings → Accelerator → T4'

# ── Locate uploaded dataset ───────────────────────────────────────────────────
# Kaggle mounts datasets at /kaggle/input/<dataset-slug>/
# Change DATASET_SLUG to match your Kaggle dataset name.
DATASET_SLUG = 'saheli'   # ← change if your dataset slug is different
SRC = f'/kaggle/input/{DATASET_SLUG}'

if not os.path.exists(SRC):
    raise FileNotFoundError(
        f'Dataset not found at {SRC}\n'
        'Add your dataset: Data → Add data → Your datasets → {DATASET_SLUG}'
    )

WORKDIR = '/kaggle/working/saheli'
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)

# Copy full project tree (preserves directory structure)
shutil.copytree(SRC, WORKDIR)

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)

print('Working directory:', os.getcwd())
print('train_unsloth.py exists:', os.path.exists('finetune/train_unsloth.py'))
print('who_anc_checklist.json  :', os.path.exists('data/who_anc_checklist.json'))

## Step 3 — Configure training

In [ ]:
import os

# ── Model ─────────────────────────────────────────────────────────────────────
os.environ['HF_MODEL_ID']   = 'unsloth/gemma-4-E4B-it'

# ── Sequence length ───────────────────────────────────────────────────────────
os.environ['MAX_SEQ_LENGTH'] = '1024'

# ── Dataset location ──────────────────────────────────────────────────────────
os.environ['DATASET_PATH'] = './finetune/datasets/saheli_anc_dataset'

# ── Output paths ──────────────────────────────────────────────────────────────
os.environ['OUTPUT_DIR'] = '/kaggle/working/outputs'
os.environ['MODEL_OUT']  = '/kaggle/working/models/saheli-gemma4-e4b'

# ── Skip GGUF export (Kaggle 20GB limit — do GGUF conversion on Colab/separately) ──
os.environ['SKIP_GGUF'] = '1'

# ── Stability flags ───────────────────────────────────────────────────────────
os.environ['TORCH_COMPILE_DISABLE']      = '1'
os.environ['UNSLOTH_COMPILE_DISABLE']    = '1'
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'

print('HF_MODEL_ID   :', os.environ['HF_MODEL_ID'])
print('MAX_SEQ_LENGTH:', os.environ['MAX_SEQ_LENGTH'])
print('DATASET_PATH  :', os.environ['DATASET_PATH'])
print('MODEL_OUT     :', os.environ['MODEL_OUT'])
print('SKIP_GGUF     :', os.environ['SKIP_GGUF'])

## Step 4 — Prepare dataset + run training

This runs two scripts:

1. `finetune/prepare_dataset.py` — generates 1,000 hybrid-format WHO ANC conversations (JSON tool call + plain-language RED/YELLOW/GREEN summary), with varied prompt templates, vitals, and paraphrases.
2. `finetune/train_unsloth.py` — loads the dataset, loads Gemma 4 E4B with `FastModel` (4-bit), applies LoRA (`r=16, alpha=32`) on language layers, trains 3 epochs with `train_on_responses_only`, exports GGUF Q4_K_M.

Watch the `CHECK FORMATTED SAMPLE` output near the start of training — it must contain `<start_of_turn>user` and `<start_of_turn>model`. If not, the response-only masking will silently fail.


In [ ]:
import subprocess, sys

# Step 4a — generate dataset
prep_cmd = [sys.executable, 'finetune/prepare_dataset.py']
print('Running:', ' '.join(prep_cmd))
print('=' * 60)
ret = subprocess.call(prep_cmd, cwd='/kaggle/working/saheli')
if ret != 0:
    raise RuntimeError('Dataset generation failed')

# Step 4b — run training
train_cmd = [sys.executable, 'finetune/train_unsloth.py']
print('Running:', ' '.join(train_cmd))
print('=' * 60)
proc = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd='/kaggle/working/saheli'
)
for line in proc.stdout:
    print(line, end='', flush=True)
ret = proc.wait()

print('=' * 60)
print('Exit code:', ret)
if ret != 0:
    raise RuntimeError('Training failed — check output above.')
print('Training complete!')


## Step 5 — Verify output files

In [ ]:
import os

MODEL_OUT = '/kaggle/working/models/saheli-gemma4-e4b'

print('Output files:')
for root, _, files in os.walk(MODEL_OUT):
    for f in files:
        full = os.path.join(root, f)
        size = os.path.getsize(full) / 1e6
        print(f'  {os.path.relpath(full, MODEL_OUT):50s} {size:8.1f} MB')

# Check for GGUF
gguf_files = [f for f in os.listdir(MODEL_OUT) if f.endswith('.gguf')]
if gguf_files:
    print(f'\nGGUF file(s): {gguf_files}')
    print('Ready for llama.cpp offline inference.')
else:
    print('\nNo GGUF found — GGUF export may have failed.')
    print('Adapter weights are saved and can be pushed to HuggingFace as-is.')

## Step 6 — Package for download

In [ ]:
import os, zipfile

MODEL_OUT = '/kaggle/working/models/saheli-gemma4-e4b'
ZIP_PATH  = '/kaggle/working/saheli-gemma4-e4b.zip'

if os.path.exists(MODEL_OUT):
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(MODEL_OUT):
            for f in files:
                full = os.path.join(root, f)
                arc  = os.path.relpath(full, MODEL_OUT)
                zf.write(full, arcname=arc)
    size_mb = os.path.getsize(ZIP_PATH) / 1e6
    print(f'Created: {ZIP_PATH}  ({size_mb:.0f} MB)')
    print('Download from: Kaggle → Output tab → Download')
else:
    print('Model folder not found:', MODEL_OUT)

## Step 7 — Push to HuggingFace (optional but needed for Unsloth prize)

Add your HuggingFace token as a Kaggle secret: **Add-ons → Secrets → HF_TOKEN**


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    print('HF token loaded from Kaggle secrets.')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    if not HF_TOKEN:
        print('No HF_TOKEN found. Skipping HuggingFace push.')
        raise SystemExit(0)

from huggingface_hub import login
login(token=HF_TOKEN)

HF_REPO = 'YOUR_HF_USERNAME/saheli-gemma4-e4b'   # ← change username

from unsloth import FastModel
# The model is already saved — just push the GGUF
MODEL_OUT = '/kaggle/working/models/saheli-gemma4-e4b'

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(folder_path=MODEL_OUT, repo_id=HF_REPO)
print(f'Pushed to: https://huggingface.co/{HF_REPO}')

## Step 8 — Evaluate (optional, needs full project)

## Step 8 — Sanity check fine-tuned adapter

Loads the saved adapter with FastModel (4-bit) and runs 5 clinical cases.
This proves the fine-tune is working **without needing the GGUF**.

In [ ]:
import re, torch
from unsloth import FastModel

MODEL_OUT = '/kaggle/working/models/saheli-gemma4-e4b'

print('Loading fine-tuned adapter for inference...')
model, tokenizer = FastModel.from_pretrained(
    model_name     = MODEL_OUT,
    max_seq_length = 1024,
    load_in_4bit   = True,
)
FastModel.for_inference(model)

SYSTEM_PROMPT = (
    'You are Saheli, a maternal health assistant for ASHA workers. '
    "Given the patient's symptoms and vitals, output a JSON tool call to "
    'assess_danger_signs followed by a short RED/YELLOW/GREEN recommendation '
    'in plain language. Use WHO Antenatal Care guidelines.'
)

def _content(text):
    return [{"type": "text", "text": text}]

CASES = [
    ('Patient is 32 weeks pregnant and has severe headache and blurred vision. BP 150/100.', 'RED'),
    ('28-week patient, baby has not moved all day.',                                         'RED'),
    ('36-week patient, mild ankle swelling. BP 130/85.',                                     'YELLOW'),
    ('ASHA visit: 24w, feeling very weak, haemoglobin 6.5.',                                 'YELLOW'),
    ('16-week patient, some nausea in the mornings.',                                        'GREEN'),
]

correct = 0
for user_msg, expected in CASES:
    messages = [
        {'role': 'system', 'content': _content(SYSTEM_PROMPT)},
        {'role': 'user',   'content': _content(user_msg)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.1, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    m = re.search(r'\b(RED|YELLOW|GREEN)\b', response.upper())
    pred = m.group(1) if m else '?'
    ok = pred == expected
    correct += int(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] expected {expected:<6} got {pred:<6} | {user_msg[:65]}")
    print(f'  {response[:200]}')

print(f'\nResult: {correct}/{len(CASES)} correct')
if correct >= 4:
    print('Fine-tune is working. Ready for GGUF export.')
elif correct >= 2:
    print('Partial success — fine-tune has some signal. Check dataset and separators.')
else:
    print('0-1/5 correct — likely a masking issue. Check CHECK FORMATTED SAMPLE output above.')

In [ ]:
import os, subprocess, sys

eval_script = 'finetune/evaluate_model.py'
if os.path.exists(eval_script):
    ret = subprocess.call([sys.executable, eval_script])
    print('Evaluation exit code:', ret)
else:
    print('evaluate_model.py not found — upload the full saheli/ project to run this.')